# Neural Network / CNN Preprocessing Workflow
### Copy-paste cheatsheet voor Neural Networks en CNN
**Volgorde:** Data inladen → Preprocessing → Augmentation → Model bouwen → Trainen → Evaluatie

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, Flatten,
    Conv2D, MaxPooling2D,
    BatchNormalization
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0

# Evaluatie
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

print('TensorFlow versie:', tf.__version__)

## 2. Data Inladen
### 2a. Afbeeldingen vanuit mappen

In [ ]:
# Mappenstructuur moet zijn:
# data/
#   train/
#     klasse1/
#     klasse2/
#   test/
#     klasse1/
#     klasse2/

IMG_SIZE   = (224, 224)  # pas aan naar jouw model
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,           # normaliseren naar 0-1
    validation_split=0.2      # 20% voor validatie
)

train_generator = train_datagen.flow_from_directory(
    'pad/naar/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',  # 'binary' voor 2 klassen
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    'pad/naar/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print('Klassen:', train_generator.class_indices)
print('Train samples:', train_generator.samples)
print('Val samples:  ', val_generator.samples)

### 2b. Afbeeldingen als numpy arrays (bv. CIFAR, MNIST)

In [ ]:
# Voorbeeld met CIFAR-10
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

# OF MNIST
# (X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

# Normaliseren naar 0-1
X_train = X_train / 255.0
X_test  = X_test  / 255.0

# Labels one-hot encoderen
NUM_CLASSES = 10  # pas aan
y_train = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test  = tf.keras.utils.to_categorical(y_test,  NUM_CLASSES)

## 3. Data Bekijken

In [ ]:
# Afbeeldingen visualiseren
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i])
    ax.set_title(f'Label: {y_train[i].argmax()}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Klasse distributie checken
unique, counts = np.unique(y_train.argmax(axis=1), return_counts=True)
plt.bar(unique, counts)
plt.xlabel('Klasse')
plt.ylabel('Aantal')
plt.title('Klasse distributie')
plt.show()

## 4. Data Augmentation (optioneel)
Kunstmatig meer trainingsdata aanmaken door afbeeldingen te transformeren.
Helpt tegen overfitting!

In [ ]:
# Augmentation bij ImageDataGenerator
train_datagen_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,        # willekeurig roteren
    width_shift_range=0.2,    # horizontaal verschuiven
    height_shift_range=0.2,   # verticaal verschuiven
    horizontal_flip=True,     # horizontaal spiegelen
    zoom_range=0.2,           # inzoomen
    validation_split=0.2
)

# Augmentation als Keras layer (modernere aanpak)
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])

## 5. Model Bouwen
### 5a. Eenvoudig Dense Neural Network (voor tabular data)

In [ ]:
# Pas input_shape aan naar jouw data
model_dense = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')  # 'sigmoid' voor binaire classificatie
])

model_dense.summary()

### 5b. CNN voor afbeeldingen

In [ ]:
# Pas input_shape aan: (hoogte, breedte, kanalen)
# kanalen: 3 voor RGB, 1 voor grayscale
model_cnn = Sequential([
    # Convolutie blok 1
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Convolutie blok 2
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Convolutie blok 3
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Fully connected
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

model_cnn.summary()

### 5c. Transfer Learning (beste resultaten met weinig data)

In [ ]:
# Voorgetraind model laden (VGG16, ResNet50, EfficientNetB0)
base_model = VGG16(
    weights='imagenet',       # voorgetraind op ImageNet
    include_top=False,        # zonder de laatste classificatie laag
    input_shape=(224, 224, 3)
)

# Base model bevriezen (gewichten niet aanpassen)
base_model.trainable = False

# Eigen classificatie lagen toevoegen
model_transfer = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

model_transfer.summary()

## 6. Model Compileren
**Optimizer:** adam (standaard, bijna altijd goed)

**Loss:**
- `categorical_crossentropy` → meerdere klassen (one-hot encoded)
- `sparse_categorical_crossentropy` → meerdere klassen (integers)
- `binary_crossentropy` → 2 klassen

In [ ]:
model_cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## 7. Callbacks
**EarlyStopping:** stop trainen als model niet meer verbetert

**ModelCheckpoint:** sla het beste model op

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',   # stop als val_loss niet daalt
        patience=5,           # wacht 5 epochs voor te stoppen
        restore_best_weights=True
    ),
    ModelCheckpoint(
        'best_model.keras',   # sla beste model op
        monitor='val_accuracy',
        save_best_only=True
    )
]

## 8. Model Trainen

In [ ]:
# Met numpy arrays
history = model_cnn.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks
)

# OF met ImageDataGenerator
# history = model_cnn.fit(
#     train_generator,
#     epochs=30,
#     validation_data=val_generator,
#     callbacks=callbacks
# )

## 9. Training Visualiseren

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy curve
axes[0].plot(history.history['accuracy'],     label='Train accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val accuracy')
axes[0].set_title('Accuracy')
axes[0].legend()

# Loss curve
axes[1].plot(history.history['loss'],     label='Train loss')
axes[1].plot(history.history['val_loss'], label='Val loss')
axes[1].set_title('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

# Overfit check:
# train accuracy veel hoger dan val accuracy → overfit
# beide stijgen samen → goed model

## 10. Evaluatie

In [ ]:
# Test score
test_loss, test_acc = model_cnn.evaluate(X_test, y_test)
print(f'Test accuracy: {test_acc:.4f}')
print(f'Test loss:     {test_loss:.4f}')

In [ ]:
# Voorspellingen
y_pred_prob = model_cnn.predict(X_test)
y_pred      = y_pred_prob.argmax(axis=1)
y_true      = y_test.argmax(axis=1)

# Classification report
print(classification_report(y_true, y_pred))

# Confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

## 11. Model Opslaan en Laden

In [ ]:
# Model opslaan
model_cnn.save('mijn_model.keras')

# Model laden
model_geladen = tf.keras.models.load_model('mijn_model.keras')

## 12. Veelgebruikte Activatiefuncties
| Functie | Gebruik |
|---|---|
| `relu` | verborgen lagen (standaard keuze) |
| `sigmoid` | output laag binaire classificatie |
| `softmax` | output laag meerdere klassen |
| `linear` | output laag regressie |

## Veelgebruikte Parameters
| Parameter | Uitleg |
|---|---|
| `epochs` | aantal keer dat model de traindata ziet |
| `batch_size` | aantal samples per update (32 of 64 is standaard) |
| `dropout` | willekeurig neuronen uitschakelen tegen overfit (0.2-0.5) |
| `learning_rate` | hoe snel het model leert (standaard 0.001) |